In [1]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
FDIC_API = "https://banks.data.fdic.gov/api"

In [3]:
def scrape_fdic_financials(limit: int = 10000) -> pd.DataFrame:
    """Scrape FDIC financial data for public banks."""
    url = f"{FDIC_API}/financials"
    params = {
        "filters": "REPDTE:20221231",  # Reporting date
        "fields": (
            "REPDTE,CERT,INSTNAME,CITY,STALP,ASSET,DEP,LNLSNET,"
            "P3ASSET,P9ASSET,NCLNLS,NITEFYQ,ROA,ROE,EEFFR,ERESSION"
        ),
        "limit": limit,
        "offset": 0,
        "sort_by": "ASSET",
        "sort_order": "DESC",
    }

    print("[SCRAPING] FDIC Financial Data...")
    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()["data"]
    records = [item["data"] for item in data]
    df = pd.DataFrame(records)
    print(f"  → {len(df):,} bank records scraped")
    return df


def scrape_fdic_failures() -> pd.DataFrame:
    """Scrape list of failed banks from FDIC."""
    url = f"{FDIC_API}/failures"
    params = {
        "fields": "CERT,INSTNAME,CITY,ST,FAILDATE,COST,RESTYPE,QBFASSET",
        "limit": 5000,
        "sort_by": "FAILDATE",
        "sort_order": "DESC",
    }

    print("[SCRAPING] FDIC Bank Failures...")
    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()["data"]
    records = [item["data"] for item in data]
    df = pd.DataFrame(records)
    print(f"  → {len(df):,} failure records scraped")
    return df

In [4]:
df_fdic_fin = scrape_fdic_financials()
df_fdic_fail = scrape_fdic_failures()

[SCRAPING] FDIC Financial Data...
  → 4,773 bank records scraped
[SCRAPING] FDIC Bank Failures...
  → 4,117 failure records scraped


In [5]:
print("=" * 60)
print(f"FDIC Financials: {df_fdic_fin.shape}")
print(f"FDIC Failures:   {df_fdic_fail.shape}")
print("=" * 60)

print("\n--- Preview Financials ---")
display(df_fdic_fin.head(3))

print("\n--- Preview Failures ---")
display(df_fdic_fail.head(3))

FDIC Financials: (4773, 14)
FDIC Failures:   (4117, 7)

--- Preview Financials ---


,ROA,REPDTE,ROE,ASSET,DEP,NCLNLS,CITY,P9ASSET,P3ASSET,CERT,STALP,LNLSNET,EEFFR,ID
0,1.029743,20221231,11.45,3201942000,2440722000,8569000,COLUMBUS,1534000,5512000,628,OH,1124240000,56.984389,628_20221231
1,1.114429,20221231,11.91,2418508000,2042255000,5534000,CHARLOTTE,1704000,4356000,3510,NC,1029699000,56.228497,3510_20221231
2,0.896086,20221231,9.35,1764139000,1399631000,4189000,SIOUX FALLS,2552000,4222000,7213,SD,625986000,56.941598,7213_20221231



--- Preview Failures ---


,COST,QBFASSET,CITY,FAILDATE,CERT,RESTYPE,ID
0,5671.0,72947.0,LENEXA,7/17/2026,25744.0,FAILURE,4117
1,1194.0,3808.0,KENTLAND,7/10/2026,28722.0,FAILURE,4116
2,97284.0,305716.0,LAGRANGE,5/1/2026,25796.0,FAILURE,4115


In [ ]:
output_dir = "../A. Data Pipeline/Data/bronze/"
os.makedirs(output_dir, exist_ok=True)

df_fdic_fin.to_csv(os.path.join(output_dir, "fdic_financials_raw.csv"), index=False)
df_fdic_fail.to_csv(os.path.join(output_dir, "fdic_failures_raw.csv"), index=False)

print("Bronze: FDIC data saved successfully.")

Bronze: FDIC data saved successfully.
